*메타코드 부트캠프 3주차 모델링 기초 강의의 필기노트 정리본이며, 강의자료를 참고하여 만들었습니다*

# Decision Tree

알고리즘은 단순한 규칙이다. 우리는 알고리즘의 동작원리를 이해해야 최선의 의사결정과 판단을 할 수 있다.

오늘 리뷰할 Decision Tree는 현업에서 많이 활용되는 알고리즘(또는 이것을 학습시키면 모형)이다. 

---

## Decision Tree의 장단점

**장점**  
1. DT는 별도의 GPU 서버 없이, 기존에 존재하는 운영 DB인 RDB에 그대로 내재화(빌트인)이 가능하다. 즉, 모형을 SQL 형태로 만들어서 DB에 bulit-in 가능하다. 그래서 가성비가 좋은 알고리즘이다.  
> IF else는 RDB에 쿼리형태로 그대로 만들어서 사용할 수 있기 때문에!  
> 또한, 굉장히 예측이 빠르다(낮은 계산량을 요구하기 때문에)

-> 이런 특징으로 비교적 오래된 알고리즘이지만, 현업에서도 아직 많이 사용한다.(딥러닝 모델의 경우 GPU 서버가 필요하며, 이 서버에 딥러닝 모형을 두고 써야한다. 즉, DL 모형이 DB와 분리되어 있다.)  

2. Excel을 다루는 업무에 최적화된모델
작은 테이블 형태의 데이터(tabular data 마치 df같은..)에 특화된 모델이다.  

3. 결측값 처리 없이도 모형이 잘 돌아간다.  

4. 인간이 잘 이해할 수 있도록 레포트를 만들 수 있다! -> 앙상블 기법 모형만 가더라도 ML 모델의 블랙박스 문제가 있다. 하지만, DT는 우리가 시각화를 통해서 알고리즘의 학습 방식을 보고 쉽게 이해할 수 있다.  

5. 그 밖에, 작은 데이터에서 강함, 변수 타입에 유연성, 스케일의 불변성 등등...

*DL과 비교해보자면,*   

DL은 업무에서 데이터양이 굉장히 커야하며, 이 데이터를 학습시키기 위한 GPU 인프라가 필요하다. 이러한 제약 때문에 업무를 하시는 분들이 신경망 모델 사용이 굉장히 어렵다. 이에 반해 DT는 로컬 컴퓨터에서 동작 가능함. input data도 엑셀 그 자체를 받을 수 있어서 편하다. 


**단점**

과적합의 여지가 크며, 이미지나 텍스트에 취약하다. 또한 표현력이 낮다. 

---

## Decision-Tree의 개념
---
### DT의 분할여부 결정  
**핵심**  

데이터가 최대한 비슷해지도록 분할하는 것!  우리는 여기서 비슷한 정도를 '불순도'라고 정의한다. 당연히 불순도가 낮으면 비슷한 애들끼리 있는 것이다.

하지만.. 이런식의 정의를 사용할 수는 없기에 불순도를 정의하기 위해 ‘Entropy’라는 물리학공식을 빌려서 사용한다. 

*Entropy: 집단이 불순하면 값이 커지고 집단이 순수하면 값이 작아지는 특성이다. 즉 엔트로피(Entropy)가 크다는 것은 시스템 내의 무질서도(disorder)가 높다는 것. 그러므로 불순도를 낮추는 즉, 엔트로피를 낮추는 방식으로 분할여부가 결정된다(물론 이건 알고리즘 학습 작동 원리임)*

분할할 수 있는 경우의 수는 물론 많다. 그 중 최적의 분할 기준을 찾기 위해서 엔트로피를 계산하는 것이다.

DT는 분할 전과 분할 후의 엔트로피 차이를 통해서 분할여부를 결정하게 된다.   
> 엔트로피 0은 모두 같은 값이라는 것이다. 더이상 순수해질 수 없는 상태이다. 

---
### 꼭 엔트로피?  

꼭Shannon Entropy를 써야하나요?  
> 아니! 불순도를 표현하기 위해서는 다양한 수식이 있다. 엔트로피 말고 다양한 수식이 있다. 

> 단, 공통적으로 집단이 불순하면 값이 커지고 순수하면 값이 작아지는 특성만 표현 가능하면? 어떤 것이든지 가능하다. 

> 대신, 각각 장단점이 있다. 

우리는 "gini", "entropy", "log_loss"를 사용할 수 있다(HPO에서 critrion에 해당하는 부분이다)

Gini- 빠른계산, 대용량 데이터셋- 균형잡힌클래스분포  
Entropy- 분포가 불균형한 데이터셋- 과적합에강함  
log_loss- 이진분류모형- 해석의편리함(확률로해석가능)   
> 우리가 분류해야하는 클래스가 단 두가지일때만 사용이 가능하다. 이런 한계가 있지만,로그 로스로 나온 결과는 확률로 나오기 때문에 인간이 직관적으로 이해하기 쉽다. 

--- 
### Decision-Tree Process

목적: 데이터를 순수한 부분 집합으로 분할하는 것이 목표다!!

용어정리
    리프: 최종적으로 분화된 부분집합으로 리프라고 한다.

    그 중간을 샘플 혹은 노드라고 한다. 


특정 노드를 기준으로 분할전 엔트로피 계산 -> 분할 후 정보획득을 계산한다. 이떄 정보획득은 분할전 엔트로피와 분할 후 엔트로피로 계산한다. 이것이 우리가 원하는 기준(순수도 높이는 방향)을 충족한다? 그럼 두 가지로 분할한다. 

이때 두 가지는 자식노드라고 부른다. 각각의 자식노드에 대해서도 분할의 과정을 거치는 과정을 반복한다. 

이 결과 다양한 리프와 노드를 가진 모형으로 분할하게 된다. 

*정보획득: 분할전엔트로피-분할후엔트로피*  
*최적분할: 정보 획득이 가장 큰 분할 선택(왜? 정보 획득이 크다는 것은 곳 빼주는 분할 후 엔트로피가 작다는 것. 분할 후 엔트로피가 작다는 것은 곧, 불순도가 작은 방향으로 분할했다는 것을 의미하기 때문이다)